# GPS Extractor
Parse a Garmin TCX GPS log into a `time, lat, lon` csv, then align it to the source video's frames.

# Config

In [1]:
case_name = "12_walk_home_2"
experiment = None

from pathlib import Path
import sys
%load_ext autoreload
%autoreload 2
project_root = Path.cwd()
while not (project_root / "src" / "A_Config.py").exists():
    if project_root == project_root.parent:
        raise RuntimeError("couldn't find repo root (src/A_Config.py) above cwd")
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from A_Config import set_case, gps_data_dir, source_video_path, FRAME_NAME_FMT
from B_video_processing import video_start_time, frame_indices_at_times
set_case(case_name, experiment)

# Extract

In [2]:
import xml.etree.ElementTree as ET
import pandas as pd

_TCX_NS = "{http://www.garmin.com/xmlschemas/TrainingCenterDatabase/v2}"


def tcx_to_GPS_track(tcx_path=None):
    '''INPUT  : path to a Garmin TCX file (device GPS log, one Trackpoint per
                sample) -- defaults to the single *.tcx sitting in gps_data_dir(),
                so a normal call is zero-arg.
       OUTPUT : DataFrame with columns [time, lat, lon], one row per Trackpoint
                that has both a Time and a Position. Also written to
                gps_data_dir() / "GPS_track.csv".'''
    if tcx_path is None:
        tcx_path = next(gps_data_dir().glob("*.tcx"))

    root = ET.parse(tcx_path).getroot()
    rows = []
    for tp in root.iter(f"{_TCX_NS}Trackpoint"):
        time = tp.find(f"{_TCX_NS}Time")
        lat = tp.find(f"{_TCX_NS}Position/{_TCX_NS}LatitudeDegrees")
        lon = tp.find(f"{_TCX_NS}Position/{_TCX_NS}LongitudeDegrees")
        if time is None or lat is None or lon is None:
            continue
        rows.append((time.text, float(lat.text), float(lon.text)))

    df = pd.DataFrame(rows, columns=["time", "lat", "lon"])
    out_path = gps_data_dir() / "GPS_track.csv"
    df.to_csv(out_path, index=False)
    print(f"Wrote {len(df)} rows to {out_path}")
    return df

In [3]:
df = tcx_to_GPS_track()
df.head()

Wrote 124 rows to /workspace/project/Data/12_walk_home_2/000_GPS_Data/GPS_track.csv


,time,lat,lon
0,2026-08-12T14:55:07.000Z,51.557724,-0.121813
1,2026-08-12T14:55:11.000Z,51.557720,-0.121794
2,2026-08-12T14:55:28.000Z,51.557747,-0.121560
3,2026-08-12T14:55:32.000Z,51.557751,-0.121481
4,2026-08-12T14:55:38.000Z,51.557758,-0.121384


# Align to video frames

In [4]:
def GPS_track_to_frames(df, video_path=None):
    '''INPUT  : GPS track DataFrame (as returned by tcx_to_GPS_track, UTC time
                strings), path to the source video -- defaults to
                source_video_path(), so a normal call only needs df.
       OUTPUT : DataFrame with columns [Frame, lat, lon], Frame formatted with
                FRAME_NAME_FMT (e.g. "0125.jpg") to match what csv_to_GPS_dict
                expects.
                video_start_time() returns the device's local clock time
                (parsed from the VID_YYYYMMDD_HHMMSSfff filename), so it's
                localised to Europe/London and converted to UTC before
                comparing against the GPS track's UTC timestamps.
                This video is VFR, so frame index is NOT computed as
                elapsed_seconds * fps (that formula assumes even frame
                spacing and drifts on VFR footage). Instead it reuses
                frame_indices_at_times, which does a single sequential
                cap.grab() pass (no pixel decode, only indices are needed
                here) and matches each target time to the frame whose own
                decoded timestamp (CAP_PROP_POS_MSEC) is nearest -- the
                same VFR-safe approach B_video_processing already uses
                elsewhere.'''
    if video_path is None:
        video_path = source_video_path()

    start_utc = pd.Timestamp(video_start_time(video_path)).tz_localize("Europe/London").tz_convert("UTC")
    times = pd.to_datetime(df["time"], utc=True)
    target_times_s = (times - start_utc).dt.total_seconds().tolist()

    frame_idx = frame_indices_at_times(video_path, target_times_s)

    keep = [i for i, (t, idx) in enumerate(zip(target_times_s, frame_idx)) if idx is not None and t >= 0]
    out = pd.DataFrame({
        "Frame": [FRAME_NAME_FMT.format(frame_idx[i]) for i in keep],
        "lat": df["lat"].iloc[keep].values,
        "lon": df["lon"].iloc[keep].values,
    })

    out_path = gps_data_dir() / "GPS_tags.csv"
    out.to_csv(out_path, index=False)
    print(f"Wrote {len(out)} rows to {out_path}")
    return out

In [ ]:
from B_video_processing import frame_indices_at_times
frames_df = GPS_track_to_frames(df)
frames_df.head()

Wrote 124 rows to /workspace/project/Data/12_walk_home_2/000_GPS_Data/GPS_tags.csv


,Frame,lat,lon
0,0235.jpg,51.557724,-0.121813
1,0356.jpg,51.557720,-0.121794
2,0867.jpg,51.557747,-0.121560
3,0987.jpg,51.557751,-0.121481
4,1168.jpg,51.557758,-0.121384


: 